# Host your fine-tuned misconception model temporarily on Kaggle

Prepared 3 November 2026. **Template, not a verified deployment of your checkpoint.**
Supports Hugging Face causal language models (full checkpoint or causal-LM LoRA adapter) and full Hugging Face sequence classifiers. GGUF, retrieval encoders, custom rerankers and classifier adapters need a different loader. Do not switch a classifier into generation mode.

1. Upload/import this notebook into a **private** Kaggle notebook. Enable GPU and Internet in Session options. Attach your checkpoint as a private input. For LoRA, attach the exact base checkpoint and tokenizer too.
2. Fill the configuration cell, including the ORIGINAL label namespace. Eedi Kaggle IDs, Eedi graph IDs and MAP labels are different spaces. Supply the training prompt/template and training/validation question IDs before claiming independent accuracy.
3. In Kaggle Secrets create `TUTOR_API_KEY`: a random temporary key of at least 32 characters, dedicated to this endpoint. Enable that secret for this notebook. If absent, the notebook asks with hidden input. It never prints the key.
4. Run cells in order. Check authenticated `/health` and `/metadata`. The notebook creates a temporary HTTPS URL; leave the session running for testing.
5. Share the **URL** in chat. Put the temporary key locally in the ignored `09_skill_misconception_and_scope_mapping/.remote_model_connection.json` file using the example provided in this project. Do not share Kaggle account credentials or Hugging Face tokens.
6. Use the stop cell when finished. The endpoint disappears when the session/tunnel stops. This is a development connection, not permanent hosting.

No competition CSV, training example or student text is bundled. The API returns raw predictions, **not validated diagnoses**. Keep private data and notebook outputs private; use only evaluation data whose access terms cover this processing. Do not publish the notebook with live keys or private model outputs.


In [ ]:
# Uses Kaggle's installed Torch/CUDA. Record the actual package versions below.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                       'transformers', 'accelerate', 'peft', 'safetensors'])


In [ ]:
from pathlib import Path
MODEL_DIR = '/kaggle/input/REPLACE_WITH_YOUR_CHECKPOINT_FOLDER'
TASK = 'causal_lm'  # OR 'sequence_classification' for a full classifier checkpoint
IS_LORA = False
BASE_MODEL_DIR = ''  # Exact locally attached base checkpoint for a causal-LM LoRA
TOKENIZER_DIR = MODEL_DIR  # Change to attached base tokenizer if the adapter has none
MODEL_LABEL = 'REPLACE_WITH_MODEL_NAME_AND_TRAINING_RUN'
LABEL_NAMESPACE = 'REPLACE_WITH_eedi_kaggle_OR_eedi_graph_OR_map_OR_YOUR_NAMESPACE'
LABEL_MAP_FILE = ''  # Classifier only: JSON {"0": "original_label_or_id", ...}; or use config.id2label
MAX_CONTEXT_TOKENS = 2048  # Never silently truncate; reduce if the checkpoint has a smaller context
ADD_SPECIAL_TOKENS_FOR_RAW_PROMPT = True  # Match how your training prompt was tokenized

assert TASK in {'causal_lm', 'sequence_classification'}
assert not (IS_LORA and TASK != 'causal_lm'), 'Classifier adapters need the matching training loader.'
assert 'REPLACE' not in MODEL_DIR + MODEL_LABEL + LABEL_NAMESPACE, 'Fill configuration first.'
for directory in [MODEL_DIR, TOKENIZER_DIR] + ([BASE_MODEL_DIR] if IS_LORA else []):
    resolved = Path(directory).resolve()
    assert resolved.is_relative_to(Path('/kaggle/input')) and resolved.is_dir(), directory
assert type(MAX_CONTEXT_TOKENS) is int and 128 <= MAX_CONTEXT_TOKENS <= 4096

if IS_LORA:
    assert (Path(MODEL_DIR) / 'adapter_model.safetensors').is_file(), 'Attach safetensors adapter weights.'


In [ ]:
import hashlib, importlib.metadata, json, torch
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
assert torch.cuda.is_available(), 'Enable GPU before loading this model.'
# Stop an earlier endpoint before replacing its model.
if 'api_server' in globals():
    api_server.shutdown(); api_server.server_close(); del api_server
if 'tunnel_process' in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate(); tunnel_process.wait(timeout=10)

load_options = dict(device_map='auto', dtype=torch.float16, local_files_only=True,
                    trust_remote_code=False, use_safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, local_files_only=True, trust_remote_code=False)
loader = AutoModelForCausalLM if TASK == 'causal_lm' else AutoModelForSequenceClassification
model = loader.from_pretrained(BASE_MODEL_DIR if IS_LORA else MODEL_DIR, **load_options)
if IS_LORA:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, MODEL_DIR, is_trainable=False, local_files_only=True)
model.eval()
configuration = getattr(model.config, 'text_config', model.config)
model_limit = getattr(configuration, 'max_position_embeddings', MAX_CONTEXT_TOKENS)
MAX_CONTEXT_TOKENS = min(MAX_CONTEXT_TOKENS, model_limit)
labels = None
if TASK == 'sequence_classification':
    labels = json.loads(Path(LABEL_MAP_FILE).read_text()) if LABEL_MAP_FILE else model.config.id2label
    labels = {int(k): str(v) for k, v in labels.items()}
    assert set(labels) == set(range(model.config.num_labels)), 'Incomplete classifier label mapping'
    assert len(set(labels.values())) == len(labels), 'Duplicate classifier labels'
    assert not any(v.startswith('LABEL_') for v in labels.values()), 'Provide original IDs/labels, not generic LABEL_0.'

# Hash real local checkpoint/tokenizer files; tokens and inference inputs are not saved.
file_manifest = {}
for directory in sorted(set([MODEL_DIR, TOKENIZER_DIR] + ([BASE_MODEL_DIR] if IS_LORA else []))):
    for file in sorted(Path(directory).rglob('*')):
        if file.is_file() and file.suffix in {'.json', '.safetensors', '.model', '.txt'}:
            with file.open('rb') as stream:
                digest = hashlib.file_digest(stream, 'sha256').hexdigest()
            file_manifest[str(file)] = dict(bytes=file.stat().st_size, sha256=digest)
Path('/kaggle/working/checkpoint_manifest.json').write_text(json.dumps(file_manifest, indent=2))
metadata = dict(model=MODEL_LABEL, task=TASK, label_namespace=LABEL_NAMESPACE, is_lora=IS_LORA,
                context_limit=MAX_CONTEXT_TOKENS, diagnosis_validated=False,
                checkpoint_manifest_sha256=hashlib.sha256(json.dumps(file_manifest,sort_keys=True).encode()).hexdigest(),
                versions={name: importlib.metadata.version(name) for name in ['torch','transformers','accelerate','peft']})
Path('/kaggle/working/inference_environment.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))


In [ ]:
import time

def infer(body):
    started = time.perf_counter()
    chat = 'messages' in body
    if chat:
        if TASK != 'causal_lm' or not tokenizer.chat_template:
            raise ValueError('This checkpoint needs its exact raw training prompt.')
        text = tokenizer.apply_chat_template(body['messages'], tokenize=False, add_generation_prompt=True)
    else:
        text = body['prompt']  # Send the EXACT training-time format; no diagnostic prompt is invented here.
    inputs = tokenizer(text, return_tensors='pt', truncation=False,
                       add_special_tokens=False if chat else ADD_SPECIAL_TOKENS_FOR_RAW_PROMPT)
    count = inputs['input_ids'].shape[-1]
    reserve = body['max_new_tokens'] if TASK == 'causal_lm' else 0
    if count < 1 or count + reserve > MAX_CONTEXT_TOKENS:
        raise ValueError('Context limit exceeded')
    inputs = {k: v.to(model.device) for k,v in inputs.items()}
    with torch.inference_mode():
        if TASK == 'causal_lm':
            output = model.generate(**inputs, max_new_tokens=reserve, do_sample=False, max_time=60,
                                    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id)
            generated = output[0, count:]
            result = dict(text=tokenizer.decode(generated, skip_special_tokens=True),
                          generated_tokens=len(generated), output_limit_reached=len(generated) >= reserve)
        else:
            probabilities = model(**inputs).logits[0].float().softmax(-1)
            scores, indexes = probabilities.topk(min(5, len(labels)))
            result = dict(candidates=[dict(label=labels[int(i)], raw_model_probability=float(s)) for i,s in zip(indexes,scores)],
                          probabilities_calibrated=False)
    return dict(**result, model=MODEL_LABEL, label_namespace=LABEL_NAMESPACE, input_tokens=count,
                seconds=time.perf_counter()-started, diagnosis_confirmed=False)


In [ ]:
# Exact source shared with the locally tested endpoint module.
API_SOURCE = '"""Small authenticated, non-streaming inference endpoint for a temporary notebook."""\nimport json\nimport secrets\nimport threading\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\n\ndef validate_request(body):\n    if not isinstance(body, dict) or set(body) - {"prompt", "messages", "max_new_tokens"}:\n        raise ValueError("Use prompt OR messages, plus optional max_new_tokens")\n    if ("prompt" in body) == ("messages" in body):\n        raise ValueError("Provide exactly one of prompt or messages")\n    maximum = body.get("max_new_tokens", 96)\n    if type(maximum) is not int or not 1 <= maximum <= 256:\n        raise ValueError("max_new_tokens must be an integer from 1 to 256")\n    if "prompt" in body:\n        if not isinstance(body["prompt"], str) or not body["prompt"].strip() or len(body["prompt"]) > 16000:\n            raise ValueError("prompt must contain 1–16000 characters")\n    else:\n        messages = body["messages"]\n        if not isinstance(messages, list) or not 1 <= len(messages) <= 32:\n            raise ValueError("messages must contain 1–32 entries")\n        for message in messages:\n            if (not isinstance(message, dict) or set(message) != {"role", "content"}\n                    or message["role"] not in ("system", "user", "assistant")\n                    or not isinstance(message["content"], str) or not message["content"].strip()):\n                raise ValueError("Invalid chat message")\n        if sum(len(m["content"]) for m in messages) > 16000:\n            raise ValueError("Chat exceeds 16000 characters")\n    return {**body, "max_new_tokens": maximum}\n\n\ndef start_server(infer, metadata, api_key, port=8000):\n    if not isinstance(api_key, str) or len(api_key) < 32 or not api_key.isascii() or not api_key.isprintable():\n        raise ValueError("Use a random ASCII API key of at least 32 characters")\n    busy = threading.Lock()\n\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self, *args):\n            pass  # Keep prompts and authentication out of notebook/access logs.\n\n        def send_json(self, status, value):\n            payload = json.dumps(value, ensure_ascii=False, allow_nan=False).encode("utf-8")\n            self.send_response(status)\n            self.send_header("Content-Type", "application/json; charset=utf-8")\n            self.send_header("Content-Length", str(len(payload)))\n            self.send_header("Cache-Control", "no-store")\n            self.end_headers()\n            try:\n                self.wfile.write(payload)\n            except (BrokenPipeError, ConnectionResetError):\n                pass\n\n        def authorized(self):\n            supplied = self.headers.get("Authorization", "").encode("utf-8")\n            if not secrets.compare_digest(supplied, ("Bearer " + api_key).encode("utf-8")):\n                self.send_json(401, {"error": "Unauthorized"})\n                return False\n            return True\n\n        def do_GET(self):\n            if not self.authorized():\n                return\n            if self.path == "/health":\n                self.send_json(200, {"status": "ok"})\n            elif self.path == "/metadata":\n                self.send_json(200, metadata)\n            else:\n                self.send_json(404, {"error": "Not found"})\n\n        def do_POST(self):\n            self.connection.settimeout(15)\n            if not self.authorized():\n                return\n            if self.path != "/predict":\n                self.send_json(404, {"error": "Not found"})\n                return\n            if self.headers.get_content_type() != "application/json" or self.headers.get("Transfer-Encoding"):\n                self.send_json(415, {"error": "Send a JSON body with Content-Length"})\n                return\n            try:\n                length = int(self.headers.get("Content-Length", "0"))\n                if not 0 < length <= 65536:\n                    self.send_json(413, {"error": "JSON body must be 1–65536 bytes"})\n                    return\n                payload = self.rfile.read(length)\n                if len(payload) != length:\n                    raise ValueError("Incomplete JSON body")\n                body = validate_request(json.loads(payload))\n            except (ValueError, UnicodeError, OSError):\n                self.send_json(400, {"error": "Invalid request schema or body"})\n                return\n            if not busy.acquire(blocking=False):\n                self.send_json(429, {"error": "Model busy; retry later"})\n                return\n            try:\n                result = infer(body)\n                self.send_json(200, result)\n            except ValueError:\n                self.send_json(422, {"error": "Unsupported input format or model limit exceeded; nothing truncated"})\n            except Exception as exc:\n                print("Inference failed:", type(exc).__name__, flush=True)\n                self.send_json(500, {"error": "Inference failed; check model configuration in notebook"})\n            finally:\n                busy.release()\n\n    server = ThreadingHTTPServer(("127.0.0.1", port), Handler)\n    thread = threading.Thread(target=server.serve_forever, daemon=True)\n    thread.start()\n    return server\n'
exec(compile(API_SOURCE, "kaggle_inference_api.py", "exec"))


In [ ]:
import getpass, urllib.request
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret('TUTOR_API_KEY')
except Exception:
    api_key = getpass.getpass('Temporary endpoint key (at least 32 random characters): ')
api_server = start_server(infer, metadata, api_key, port=8000)
http = urllib.request.build_opener(urllib.request.ProxyHandler({}))
request = urllib.request.Request('http://127.0.0.1:8000/health', headers={'Authorization': 'Bearer ' + api_key})
with http.open(request, timeout=5) as response:
    assert json.load(response)['status'] == 'ok'
print('Local authenticated health check passed. Key was not printed.')


## Publish only the temporary authenticated endpoint
The next cell downloads the official Cloudflare binary and verifies the release asset SHA-256 supplied by GitHub. If a digest is unavailable, it stops. The API itself checks a Bearer key on **every** route and serializes inference. It does not expose notebook files or account APIs. Requests are JSON, not streaming. Cloudflare sees traffic passing through its service; use only data permitted for this processing.


In [ ]:
import os, re, subprocess, time, urllib.error
# A failed earlier tunnel attempt stops the local server. Check it before starting another tunnel.
try:
    local_request = urllib.request.Request('http://127.0.0.1:8000/health', headers={'Authorization': 'Bearer ' + api_key})
    with http.open(local_request, timeout=5) as response:
        if json.load(response).get('status') != 'ok':
            raise RuntimeError('Unexpected local server response')
except (NameError, urllib.error.URLError, TimeoutError, OSError) as error:
    raise RuntimeError('Local model server is not ready. Rerun the preceding cell containing api_server = start_server(...), then rerun this tunnel cell.') from error
print('Local authenticated server is ready.')
release_request = urllib.request.Request('https://api.github.com/repos/cloudflare/cloudflared/releases/latest',
                                         headers={'User-Agent': 'ncert-local-evaluation'})
with urllib.request.urlopen(release_request, timeout=30) as response:
    release = json.load(response)
asset = next(a for a in release['assets'] if a['name'] == 'cloudflared-linux-amd64')
expected = asset.get('digest', '')
assert re.fullmatch(r'sha256:[a-f0-9]{64}', expected), 'No publisher asset digest; do not run an unverified binary.'
assert asset['browser_download_url'].startswith('https://github.com/cloudflare/cloudflared/releases/download/')
binary = Path('/kaggle/working/cloudflared')
with urllib.request.urlopen(asset['browser_download_url'], timeout=60) as response, binary.open('wb') as output:
    while data := response.read(1024*1024):
        output.write(data)
with binary.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected.split(':')[1], 'Binary hash mismatch'
binary.chmod(0o700)
log_path = Path('/kaggle/working/tunnel.log')
with log_path.open('w') as log:
    tunnel_process = subprocess.Popen([str(binary), 'tunnel', '--url', 'http://127.0.0.1:8000',
                                       '--protocol', 'http2', '--no-autoupdate'], stdout=log, stderr=subprocess.STDOUT)
try:
    endpoint = None
    for _ in range(120):
        if tunnel_process.poll() is not None:
            raise RuntimeError('Tunnel exited; inspect tunnel.log (it contains no API key).')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_path.read_text())
        if match:
            endpoint = match.group(); break
        time.sleep(.5)
    if endpoint is None:
        raise TimeoutError('No tunnel URL after 60 seconds')
    # The public DNS record and tunnel route may take time to propagate.
    remote_request = urllib.request.Request(endpoint + '/health', headers={'Authorization': 'Bearer ' + api_key})
    deadline = time.monotonic() + 90
    last_error = None
    while time.monotonic() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError('Tunnel exited; inspect tunnel.log (it contains no API key).')
        try:
            with http.open(remote_request, timeout=5) as response:
                if json.load(response).get('status') != 'ok':
                    raise RuntimeError('Public health response was unexpected')
            break
        except urllib.error.HTTPError as error:
            if error.code not in (502, 503, 504):
                raise  # In particular, never ignore a failed authentication check.
            last_error = error
        except (urllib.error.URLError, TimeoutError, OSError) as error:
            last_error = error
        time.sleep(2)
    else:
        raise RuntimeError('Public tunnel did not become reachable after 90 seconds; check Kaggle Internet and tunnel.log') from last_error
    try:
        http.open(endpoint + '/health', timeout=10)
        raise RuntimeError('Unauthenticated request unexpectedly succeeded')
    except urllib.error.HTTPError as error:
        assert error.code == 401, error.code
    print('Temporary URL:', endpoint)
    print('cloudflared release:', release['tag_name'], expected)
    print('Share the URL; keep the key only in your local ignored connection file. Leave this session running.')
except BaseException:
    tunnel_process.terminate(); tunnel_process.wait(timeout=10)
    api_server.shutdown(); api_server.server_close()
    raise


## Calling the model
All routes require `Authorization: Bearer <temporary key>`.

- `GET /health`: availability only.
- `GET /metadata`: task, model identity and original label namespace.
- `POST /predict`: `{"prompt": "<your exact training-formatted input>", "max_new_tokens": 96}`; chat models can instead accept `messages` with `role` and `content`.

Do not send both prompt and messages. Classifiers require the original raw training input. Context overflow is rejected; input is never silently truncated. Outputs remain unvalidated predictions. Generative output still needs parsing, ID whitelisting, abstention checks and independent evaluation before diagnosis. A softmax number is not calibrated confidence. HTTP 429 means another request is running.

This notebook has no automatic keep-alive, reconnection or permanent deployment mechanism. Stop after the authorized evaluation.


In [ ]:
# STOP cell: run only when the evaluation session is finished.
if 'tunnel_process' in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate(); tunnel_process.wait(timeout=10)
if 'api_server' in globals():
    api_server.shutdown(); api_server.server_close(); del api_server
api_key = None
print('Endpoint stopped. Delete or rotate the temporary Kaggle secret before reusing the notebook.')


References: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [Kaggle Secrets implementation](https://github.com/Kaggle/docker-python/blob/main/patches/kaggle_secrets.py), [Transformers chat templates](https://huggingface.co/docs/transformers/main/chat_templating), [PEFT loading](https://huggingface.co/docs/peft/package_reference/auto_class), [Cloudflare temporary tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/).
